# 03 — Gold: LAD Mapping & Enrichment
Spatially joins each listing to its Local Authority District (LAD), then joins LAD-level enrichment data (house prices, GP surgeries, parks).

**Catalog:** `airbnb_app`  
**Reads from:** `airbnb_app.clean`  
**Writes to:** `airbnb_app.gold`  

| Step | What happens |
|------|-------------|
| 1 | Load LAD boundary GeoJSON |
| 2 | Load clean listings (lat/lng) per city |
| 3 | Spatial join — point-in-polygon assigns `lad_code` + `lad_name` to each listing |
| 4 | Join house prices on `lad_code` |
| 5 | Join GP surgeries and parks on `lad_code` |
| 6 | Write enriched listings to `airbnb_app.gold.airbnb_listings_{city}` |

> **Note:** This runs on the driver (single node) because GeoPandas cannot run on distributed Spark workers. For very large listing counts, consider sampling during development.

## 0. Config

In [0]:
CLEAN_DB = "airbnb_app.clean"
GOLD_DB  = "airbnb_app.gold"

CITIES = ["london", "manchester", "edinburgh", "bristol"]

# LAD boundary file
LAD_BOUNDARY_PATH = "/Volumes/airbnb_app/raw/boundary_data/Local_Authority_Districts_December_2024_Boundaries_UK_BGC_-5461244619642504325.geojson"
# Column names in the LAD boundary GeoJSON — update if different
# Common ONS naming: LAD24CD, LAD24NM (year prefix changes per release)
LAD_CODE_COL = "LAD24CD"
LAD_NAME_COL = "LAD24NM"

## 1. Setup

In [0]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "geopandas", "shapely", "--quiet"])

In [0]:
spark.sql("SELECT 1").collect()

In [0]:
import os
import pandas as pd
import geopandas as gpd
from pyspark.sql import functions as F

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_DB}")
print(f"Schema ready: {GOLD_DB}")

gold_log = []

## 2. Check LAD boundary file exists

In [0]:
if os.path.exists(LAD_BOUNDARY_PATH):
    size_mb = os.path.getsize(LAD_BOUNDARY_PATH) / 1_048_576
    print(f"✓ Found: {LAD_BOUNDARY_PATH} ({size_mb:.1f} MB)")
else:
    raise FileNotFoundError(
        f"LAD boundary file not found at {LAD_BOUNDARY_PATH}\n"
        f"Download from: https://geoportal.statistics.gov.uk "
        f"(search 'Local Authority Districts Boundaries UK BGC')"
    )

## 3. Load and inspect LAD boundaries

In [0]:
lad_gdf = gpd.read_file(LAD_BOUNDARY_PATH)

print(f"Shape: {lad_gdf.shape}")
print(f"CRS: {lad_gdf.crs}")
print(f"Columns: {list(lad_gdf.columns)}")
lad_gdf.head(3)

## 4. Standardise LAD boundaries

Confirm the code/name columns match config, reproject to WGS84 if needed, and keep only the columns we need.

In [0]:
# Reproject to WGS84 (EPSG:4326) if not already — Airbnb lat/lng use this system
if lad_gdf.crs is None:
    print("WARNING: No CRS set on boundary file — assuming EPSG:4326")
    lad_gdf = lad_gdf.set_crs("EPSG:4326")
elif lad_gdf.crs.to_epsg() != 4326:
    print(f"Reprojecting from {lad_gdf.crs} to EPSG:4326...")
    lad_gdf = lad_gdf.to_crs("EPSG:4326")
else:
    print("Already EPSG:4326 — no reprojection needed")

# Verify config column names exist
if LAD_CODE_COL not in lad_gdf.columns or LAD_NAME_COL not in lad_gdf.columns:
    raise KeyError(
        f"Expected columns '{LAD_CODE_COL}' and '{LAD_NAME_COL}' not found.\n"
        f"Available columns: {list(lad_gdf.columns)}\n"
        f"Update LAD_CODE_COL and LAD_NAME_COL in the config cell."
    )

# Keep only what we need
lad_gdf = lad_gdf[[LAD_CODE_COL, LAD_NAME_COL, "geometry"]].rename(columns={
    LAD_CODE_COL: "lad_code",
    LAD_NAME_COL: "lad_name",
})

print(f"\nReady — {len(lad_gdf)} LAD polygons")

In [0]:
# ───────────────────────────────────────────────────────────────── Section 4b: Rail station accessibility ─────────────────────────────────────────────────────────────────
# Rail station walk-time data is at MSOA level — aggregate to LAD by median, done natively in Spark

rail_by_lad = spark.sql(f"""
    SELECT
        m.lad_code,
        percentile_approx(r.less_than_15_minute_walk, 0.5) AS pct_within_15min_rail,
        percentile_approx(r.less_than_30_minute_walk, 0.5) AS pct_within_30min_rail,
        percentile_approx(r.less_than_60_minute_walk, 0.5) AS pct_within_60min_rail
    FROM {CLEAN_DB}.amenities_rail_stations r
    JOIN (
        SELECT DISTINCT msoa_code, local_authority_code AS lad_code
        FROM {CLEAN_DB}.house_prices_msoa
    ) m
    ON r.msoa_code = m.msoa_code
    GROUP BY m.lad_code
""").toPandas()

print(f"Rail accessibility aggregated to {len(rail_by_lad):,} LADs")
rail_by_lad.head()

## 5. Helper — spatial join function

In [0]:
def assign_lad(listings_pd: pd.DataFrame) -> pd.DataFrame:
    """
    Spatially joins listings to LAD polygons using point-in-polygon.
    Expects listings_pd to have 'latitude' and 'longitude' columns.
    Returns the original DataFrame with lad_code and lad_name appended.
    """
    listings_gdf = gpd.GeoDataFrame(
        listings_pd,
        geometry=gpd.points_from_xy(listings_pd["longitude"], listings_pd["latitude"]),
        crs="EPSG:4326",
    )

    joined = gpd.sjoin(
        listings_gdf,
        lad_gdf,
        how="left",
        predicate="within",
    )

    # Drop geometry and spatial join helper columns before returning to Spark
    drop_cols = [c for c in ["geometry", "index_right"] if c in joined.columns]
    joined = joined.drop(columns=drop_cols)

    return joined


print("Helper loaded.")

## 6. Join listings to LAD, then join house prices and amenities

In [0]:
# Load LAD-level enrichment tables once — reused across all cities
house_prices_pd = spark.table(f"{CLEAN_DB}.house_prices_msoa").toPandas()
gp_pd           = spark.table(f"{CLEAN_DB}.amenities_gp_surgeries").toPandas()
parks_pd        = spark.table(f"{CLEAN_DB}.amenities_parks").toPandas()

# House prices are at MSOA level — aggregate to LAD by taking the median
house_prices_lad = (
    house_prices_pd
    .groupby("local_authority_code", as_index=False)
    .agg(
        median_house_price_2025=("median_house_price_2025", "median"),
        median_house_price_2015=("median_house_price_2015", "median"),
        price_growth_10yr=("price_growth_10yr", "median"),
    )
    .rename(columns={"local_authority_code": "lad_code"})
)

print(f"House prices aggregated to LAD: {house_prices_lad.shape}")
print(f"GP surgeries: {gp_pd.shape}")
print(f"Parks: {parks_pd.shape}")


In [0]:
for city in CITIES:
    print(f"\n{'='*50}")
    print(f"{city.upper()}")
    print(f"{'='*50}")

    src = f"{CLEAN_DB}.airbnb_listings_{city}"
    tgt = f"{GOLD_DB}.airbnb_listings_{city}"

    try:
        # Load clean listings
        listings_pd = spark.table(src).toPandas()
        print(f"  Loaded {len(listings_pd):,} listings")

        # Spatial join to LAD
        listings_pd = assign_lad(listings_pd)
        matched   = listings_pd["lad_code"].notna().sum()
        unmatched = listings_pd["lad_code"].isna().sum()
        print(f"  LAD matched:   {matched:,}")
        print(f"  LAD unmatched: {unmatched:,}")

        # Join house prices on lad_code
        listings_pd = listings_pd.merge(house_prices_lad, on="lad_code", how="left")

        # Join GP surgeries on lad_code
        gp_join = gp_pd[["lad_code", "gp_surgery_count", "gps_per_100000_people"]]
        listings_pd = listings_pd.merge(gp_join, on="lad_code", how="left")

        # Join parks on lad_code
        parks_join = parks_pd[[
            "lad_code", "total_parks_count",
            "parks_and_play_areas_per_100000_people"
        ]]
        listings_pd = listings_pd.merge(parks_join, on="lad_code", how="left")
        listings_pd = listings_pd.merge(rail_by_lad, on="lad_code", how="left")

        # Convert back to Spark and write to gold
        listings_spark = spark.createDataFrame(listings_pd)
        listings_spark = listings_spark.withColumn("_gold_created_at", F.current_timestamp())

        (
            listings_spark.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(tgt)
        )

        row_count = listings_spark.count()
        print(f"  ✓ {tgt} ({row_count:,} rows)")
        gold_log.append({"city": city, "status": "ok", "rows": row_count, "lad_matched": int(matched), "lad_unmatched": int(unmatched)})

    except Exception as e:
        print(f"  ✗ {tgt} — {e}")
        gold_log.append({"city": city, "status": "error", "error": str(e)})

## 7. Gold layer summary

In [0]:
summary = pd.DataFrame(gold_log)
display(summary)

failures = summary[summary["status"] == "error"]
if not failures.empty:
    raise RuntimeError(f"Gold layer build failed for:\n{failures[['city', 'error']].to_string()}")

print("\nGold layer (LAD mapping + enrichment) complete.")

## 8. Spot checks

In [0]:
# Confirm LAD code, house prices, and amenities all populated
spark.sql("""
    SELECT id, neighbourhood_cleansed, lad_code, lad_name,
           median_house_price_2025, price_growth_10yr,
           gp_surgery_count, total_parks_count
    FROM airbnb_app.gold.airbnb_listings_manchester
    LIMIT 10
""").display()

In [0]:
# Check unmatched rate per city — Edinburgh expected to be 100% unmatched
# for house prices/amenities (England & Wales only) but LAD code itself
# should still resolve since LAD boundaries are UK-wide
spark.sql("""
    SELECT _city,
           COUNT(*)                                          AS total_listings,
           SUM(CASE WHEN lad_code IS NULL THEN 1 END)        AS missing_lad,
           SUM(CASE WHEN median_house_price_2025 IS NULL THEN 1 END) AS missing_house_price,
           SUM(CASE WHEN gp_surgery_count IS NULL THEN 1 END) AS missing_gp_data
    FROM (
        SELECT * FROM airbnb_app.gold.airbnb_listings_london
        UNION ALL SELECT * FROM airbnb_app.gold.airbnb_listings_manchester
        UNION ALL SELECT * FROM airbnb_app.gold.airbnb_listings_edinburgh
        UNION ALL SELECT * FROM airbnb_app.gold.airbnb_listings_bristol
    )
    GROUP BY _city
    ORDER BY _city
""").display()

## Notes

- **Spatial join runs on driver.** GeoPandas cannot run on distributed Spark workers, so listings are converted to pandas before the join, then back to Spark for the Delta write. For very large cities this may be slow — monitor London specifically.
- **House prices aggregated MSOA → LAD.** The raw house price data is at MSOA level; this notebook takes the median across MSOAs within each LAD. A future iteration could join directly at MSOA level for more granularity — would need MSOA boundary GeoJSON.
- **Edinburgh will show missing house price and amenities data** — these datasets cover England and Wales only. The `lad_code` itself should still populate correctly if the boundary file is UK-wide. If Edinburgh's `lad_code` is also null, the boundary file you downloaded may only cover England/Wales — check coverage.
- **Unmatched listings** (`lad_code IS NULL` for non-Edinburgh cities) usually indicate a listing's coordinates fall just outside the boundary polygon edge, or the boundary file has a coordinate precision mismatch. A small number (<1%) is expected and acceptable.
- **Next step:** Add ward and MSOA-level joins for higher granularity, then build the investment scoring notebook on top of this gold table.